# Train a "products in a room" LoRA on Qwen-Image (Google Colab)

**Read this before you run anything — this is a much bigger model than the Stable Diffusion 1.5 notebook, and it
changes what hardware you need.**

| | Stable Diffusion 1.5 | Qwen-Image |
|---|---|---|
| Parameters | ~0.9B UNet | ~20B transformer + a ~7B VLM text encoder |
| Fits on a free Colab T4 (16GB)? | Yes | **No, not even with 4-bit quantization** |
| Minimum realistic GPU | T4 (free tier) | A100 40GB (Colab **Pro** or **Pro+**) |
| Rough training time (this notebook's defaults) | 20-40 min | **Several hours** — see Section 6 before you commit to a run |
| Training objective | predict the noise (DDPM) | predict a velocity field (flow matching) — a different, newer recipe |

**What this does, same as the SD1.5 notebook:** removes the white studio background from your supplier photos, pastes
each product into a randomly generated room, captions it from the folder name, and fine-tunes a LoRA (a small add-on,
trained on top of the frozen base model) so the model learns the *look* of your catalogue. Sections 1-5 (finding,
validating, captioning and compositing the images) are **identical** to the SD1.5 notebook — same data, same logic.

**Where this notebook differs from the SD1.5 one, and why.** Qwen-Image's internals (a 20B-parameter multimodal
diffusion transformer, trained with flow matching rather than the simple noise-prediction objective SD1.5 uses) are
new enough, and complex enough, that hand-writing the forward/backward pass ourselves — the way the SD1.5 notebook
does — risks getting a subtle detail wrong in a way that is very hard for a non-expert to debug. So instead of a
hand-written training loop, **Section 8 drives Hugging Face's own maintained Qwen-Image LoRA training script**
(from the `diffusers` library's examples) as a subprocess. You still control every setting from the CONFIG cell below;
you're just not relying on this notebook's author having gotten every tensor shape right for a model this new.

## How to run it
1. **Colab: Runtime -> Change runtime type -> A100 GPU.** This requires Colab Pro or Pro+ — Qwen-Image does not fit
   on the free-tier T4. Section 6 checks your GPU and refuses to continue on anything with less than ~38GB VRAM,
   with a clear message, rather than silently failing partway through a multi-hour run.
2. Put your image folder in Google Drive — same structure as the SD1.5 notebook (`ATRIANI/`, `CELADON/`, `LAZZONI/`
   (PDFs), `LUXUS/`...).
3. Leave `DRIVE_IMAGES_DIR = "auto"` in the **CONFIG** cell to search your Drive for the supplier folders, same as
   before.
4. **Run all**, but watch Section 6's output before the training cell (Section 8) starts — it prints your measured
   seconds/step from a short warmup and an updated time estimate for your chosen step count. If that estimate is
   longer than you want to wait, lower `MAX_TRAIN_STEPS` before letting it continue.
5. The trained LoRA is written to `DRIVE_OUTPUT_DIR`, the same way as the SD1.5 notebook, with checkpoints saved
   periodically so a Colab disconnect doesn't lose your progress.

Non-image files, unreadable images and tiny thumbnails are skipped and reported, never fatal — same as the SD1.5
notebook.


In [ ]:
# ============================== CONFIG: edit these ==============================
# Where the images are. In Colab these are Google Drive paths; locally the LOCAL_* paths are used.
DRIVE_IMAGES_DIR = "/content/drive/MyDrive/Curalina - Humber Capstone/Supplier Images"   # "auto" = search your Drive + shared drives for the supplier folders.
#                             e.g. "/content/drive/Shareddrives/<Drive name>/<folder>" or "/content/drive/MyDrive/<folder>"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/curalina_qwen_lora_output"   # the trained model is written here

LOCAL_IMAGES_DIR = "/Users/rjsalmon/Documents/Humber/misc.curalina/Supplier Images"
LOCAL_OUTPUT_DIR = "./qwen_lora_output"

# Model + training settings
BASE_MODEL_ID = "Qwen/Qwen-Image"
TRIGGER = "crlnstyle"          # same trigger word convention as the SD1.5 notebook
RESOLUTION = 512               # Qwen-Image natively prefers larger (1024+), but 512 is what keeps this inside
                                # an A100's 40GB alongside a 20B transformer; raise it only if you have more VRAM.
MAX_TRAIN_STEPS = 600          # Qwen-Image is far more expensive per step than SD1.5 (see Section 6) — this is a
                                # starting point, not the SD1.5 notebook's 1200. Section 6 tells you what it will
                                # actually cost in wall-clock time for YOUR measured speed before you commit to it.
BATCH_SIZE = 1                 # keep at 1 unless you have more than an A100-40GB; raising this is the single
                                # biggest VRAM cost in this notebook.
LEARNING_RATE = 1e-4
LORA_RANK = 8
SAVE_EVERY = 150               # checkpoint to Drive every N steps (more frequent than SD1.5's 400, since a step
                                # costs so much more here — you want to lose less progress to a disconnect).
GRADIENT_CHECKPOINTING = True
QUANTIZE_4BIT = True           # load the frozen base weights in 4-bit (bitsandbytes NF4). Turn off only if you
                                # have an 80GB GPU and want full precision; on a 40GB A100 this is required.
P_ROOM_BACKGROUND = 0.8        # share of training samples pasted into a synthetic room (rest: plain product photo)
MIN_SIDE_PX = 200              # skip thumbnails smaller than this
SEED = 42

# Debug: set to a small number (e.g. 20) to try the pipeline on a few images first, or None for everything.
LIMIT_IMAGES = None


In [ ]:
import os, sys, json, random, math, shutil, re, time, difflib
from pathlib import Path
from collections import Counter

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")


def explain_missing(path: Path) -> str:
    # Finds the deepest folder that DOES exist, lists what is in it, and suggests the closest spelling.
    p = Path(path)
    missing = []
    while not p.exists() and p != p.parent:
        missing.insert(0, p.name)
        p = p.parent
    try:
        children = sorted(os.listdir(p))
    except OSError:
        children = []
    msg = f"'{path}' was not found.\nThe part that exists is: {p}\nThe first name that is missing is: '{missing[0] if missing else ''}'\n"
    msg += f"Folders that DO exist inside {p}:\n  " + ("\n  ".join(children[:40]) if children else "(none - is this the Google account that has access?)") + "\n"
    if missing and children:
        close = difflib.get_close_matches(missing[0], children, n=3, cutoff=0.4)
        if close:
            msg += f"Did you mean: {close}\n"
    msg += "Tip: in Colab open the Files panel (folder icon, left), browse to the folder, right-click -> Copy path, and paste it into the CONFIG cell."
    return msg


SUPPLIER_MARKERS = {"atriani", "celadon", "lazzoni", "luxus"}


def find_images_root(max_depth: int = 5):
    # Looks through every shared drive and My Drive for a folder that contains at least two of the supplier folders.
    roots = []
    shared = Path("/content/drive/Shareddrives")
    if shared.is_dir():
        roots += [shared / d for d in sorted(os.listdir(shared))]
    roots.append(Path("/content/drive/MyDrive"))
    hits = []
    for base in roots:
        base_parts = len(base.parts)
        for cur, dirs, _ in os.walk(base):
            depth = len(Path(cur).parts) - base_parts
            dirs[:] = [d for d in dirs if not d.startswith(".")] if depth < max_depth else []
            if len({d.lower() for d in dirs} & SUPPLIER_MARKERS) >= 2:
                hits.append(Path(cur))
                dirs[:] = []                       # found it; do not search inside
    return hits


if IN_COLAB:
    # Uncomment if a package is missing. Colab already has torch/torchvision.
    # !pip -q install diffusers transformers accelerate peft
    from google.colab import drive
    drive.mount("/content/drive",force_remount=True)
    for root in ["/content/drive", "/content/drive/Shareddrives", "/content/drive/MyDrive"]:
        print(f"{root} ->", sorted(os.listdir(root))[:30] if os.path.isdir(root) else "(not found)")

    # Make sure to run the CONFIG cell (cell b568965d) first to set DRIVE_IMAGES_DIR correctly!
    print(f"DEBUG (Cell 6f9d2616): DRIVE_IMAGES_DIR is '{DRIVE_IMAGES_DIR}'")

    if DRIVE_IMAGES_DIR.strip().lower() == "auto":
        print("Searching your Drive for the supplier image folders (looks for LUXUS / CELADON / ATRIANI / LAZZONI)...")
        hits = find_images_root()
        if len(hits) == 1:
            IMAGES_DIR = hits[0]
            print("Found:", IMAGES_DIR)
        elif len(hits) > 1:
            raise SystemExit("Found several candidate folders - copy one into DRIVE_IMAGES_DIR:\n  " + "\n  ".join(map(str, hits)))
        else:
            raise SystemExit("Could not find a folder containing LUXUS / CELADON / ATRIANI / LAZZONI anywhere in your Drive.\n"
                             "Check you mounted the Google account that has access, and that the shared drive is listed above.")
    else:
        IMAGES_DIR = Path(DRIVE_IMAGES_DIR)
    OUTPUT_DIR = Path(DRIVE_OUTPUT_DIR)
    WORK_DIR = Path("/content/work")

else:
    IMAGES_DIR = Path(LOCAL_IMAGES_DIR)
    OUTPUT_DIR = Path(LOCAL_OUTPUT_DIR)
    WORK_DIR = Path("./work")

# Check the input folder FIRST, with a readable message, before creating anything.
print(f"DEBUG (Cell 6f9d2616): IMAGES_DIR is '{IMAGES_DIR}' before checking existence.")
if not IMAGES_DIR.is_dir():
    raise SystemExit("IMAGE FOLDER PROBLEM\n" + explain_missing(IMAGES_DIR))

*(Identical to the SD1.5 notebook from here through Section 4 - same data, same logic.)*

## 1. Find the images (and skip everything else)

We walk the whole folder tree. Only these extensions are treated as images; anything else — the Lazzoni PDFs,
`.DS_Store`, spreadsheets — is counted and skipped. Then every candidate is opened once with Pillow: files that are
corrupt, unreadable or smaller than `MIN_SIDE_PX` are dropped and listed, so training never hits a bad file.


In [ ]:
import pandas as pd
from PIL import Image, UnidentifiedImageError

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp"}
Image.MAX_IMAGE_PIXELS = 60_000_000  # guard against decompression bombs

found, skipped_non_image = [], Counter()
for root, dirs, files in os.walk(IMAGES_DIR):
    dirs[:] = [d for d in dirs if not d.startswith(".")]
    for name in files:
        if name.startswith("."):
            continue
        p = Path(root) / name
        ext = p.suffix.lower()
        if ext in IMAGE_EXTS:
            found.append(p)
        else:
            skipped_non_image[ext or "(no extension)"] += 1

print(f"{len(found)} candidate image files")
print("skipped non-image files by type:", dict(skipped_non_image) or "none")

valid, rejected = [], []
for p in sorted(found):
    try:
        with Image.open(p) as im:
            im.verify()                      # cheap integrity check
        with Image.open(p) as im:            # verify() invalidates the handle, reopen for size
            w, h = im.size
        if min(w, h) < MIN_SIDE_PX:
            rejected.append((str(p.relative_to(IMAGES_DIR)), f"too small ({w}x{h})"))
        else:
            valid.append(p)
    except (UnidentifiedImageError, OSError, ValueError, Image.DecompressionBombError) as e:
        rejected.append((str(p.relative_to(IMAGES_DIR)), f"unreadable: {type(e).__name__}"))

if LIMIT_IMAGES:
    random.Random(SEED).shuffle(valid)
    valid = sorted(valid[:LIMIT_IMAGES])

print(f"{len(valid)} usable images, {len(rejected)} rejected")
pd.DataFrame(rejected, columns=["file", "reason"]).head(15)


## 2. Captions from the folder structure

Supplier = the top-level folder (`LUXUS`, `CELADON`, ...). Product name = the folder the image sits in, with the
`001 - ` numbering and any trailing SKU removed. A keyword pass maps names to a furniture category and a sensible room
(a bed goes in a bedroom, a dining chair in a dining room). Nothing here needs the spreadsheets.


In [ ]:
CATEGORY_RULES = [  # first match wins; (category, regex, rooms it belongs in)
    ("dining chair", r"dining chair", ["dining room"]),
    ("dining table", r"dining table", ["dining room"]),
    ("accent chair", r"accent chair|armchair|lounge chair|poltrona|chair", ["living room", "bedroom"]),
    ("coffee table", r"coffee table|cocktail table", ["living room"]),
    ("side table", r"side table|end table", ["living room", "bedroom"]),
    ("bar cabinet", r"bar cabinet", ["dining room", "living room"]),
    ("sectional sofa", r"sectional|modular", ["living room"]),
    ("sofa", r"\bsofa\b|\bloveseat\b", ["living room"]),
    ("ottoman", r"ottoman", ["living room"]),
    ("bench", r"\bbench\b", ["bedroom", "living room"]),
    ("nightstand", r"night ?stand|bedside", ["bedroom"]),
    ("bed", r"\bbed\b|\bking\b|\bqueen\b", ["bedroom"]),
    ("sideboard", r"sideboard|buffet|credenza", ["dining room", "living room"]),
    ("table", r"table", ["dining room", "living room"]),
    ("console or cabinet", r"console|media|cabinet|shelf|bookcase|vanity|desk|stool", ["living room", "dining room"]),
]


def clean_name(folder_name: str) -> str:
    n = re.sub(r"^\s*\d+\s*-\s*", "", folder_name)          # "001 - Halo Nightstand" -> "Halo Nightstand"
    n = re.sub(r"\s*-\s*\d{4,}\s*$", "", n)                # Celadon "Traverse I - 21368" -> "Traverse I"
    n = n.replace("_", ":").strip()
    return n or folder_name


def describe(path: Path):
    rel = path.relative_to(IMAGES_DIR)
    parts = rel.parts
    supplier = parts[0] if len(parts) > 1 else "unknown"
    folder = parts[-2] if len(parts) > 2 else path.stem        # product folder; fall back to file name
    if folder.lower() in {"product images", supplier.lower()}:
        folder = path.stem
    name = clean_name(folder)
    if supplier.upper() == "CELADON":
        return dict(supplier=supplier, name=name, category="framed wall art", rooms=["living room", "bedroom"], is_art=True)
    for category, pattern, rooms in CATEGORY_RULES:
        if re.search(pattern, name.lower()):
            return dict(supplier=supplier, name=name, category=category, rooms=rooms, is_art=False)
    return dict(supplier=supplier, name=name, category="piece of furniture", rooms=["living room"], is_art=False)


records = []
for p in valid:
    d = describe(p)
    d["path"] = str(p)
    records.append(d)

df = pd.DataFrame(records)
print(df.groupby(["supplier", "category"]).size().to_string())
df[["supplier", "name", "category"]].sample(min(8, len(df)), random_state=SEED)


## 3. Cut out the products and build "product in a room" training images

Studio photos have a white background, which would teach the model "furniture floats on white". So we **remove the
background** (flood-fill from the image border — white areas *inside* the product, like a white sofa, are kept) and paste
the cutout into a randomly generated room: random wall colour, floor, baseboard, sometimes a window, with a soft shadow.
Artwork is hung on the wall instead of placed on the floor. Cutouts are cached in `WORK_DIR`.


In [ ]:
import numpy as np
from PIL import ImageDraw, ImageFilter

CUTOUT_DIR = WORK_DIR / "cutouts"
CUTOUT_DIR.mkdir(parents=True, exist_ok=True)


def matte_white_background(img: Image.Image, tolerance: int = 45, max_side: int = 640) -> Image.Image:
    rgba = img.convert("RGBA")
    if max(rgba.size) > max_side:
        rgba.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
    flat = Image.alpha_composite(Image.new("RGBA", rgba.size, (255, 255, 255, 255)), rgba).convert("RGB")
    work = flat.copy()
    w, h = work.size
    marker = (255, 0, 255)
    floor = 255 - tolerance
    for seed in [(0, 0), (w - 1, 0), (0, h - 1), (w - 1, h - 1), (w // 2, 0), (w // 2, h - 1), (0, h // 2), (w - 1, h // 2)]:
        px = flat.getpixel(seed)
        if all(int(c) >= floor for c in px[:3]):
            ImageDraw.floodfill(work, seed, marker, thresh=tolerance)
    arr = np.array(rgba)
    background = np.all(np.array(work) == np.array(marker), axis=2)
    arr[..., 3][background] = 0
    arr[..., 3][arr[..., 3] < 40] = 0
    alpha = Image.fromarray(arr[..., 3]).filter(ImageFilter.MinFilter(3))   # erode 1px: drops the white fringe
    arr[..., 3] = np.array(alpha)
    return Image.fromarray(arr, "RGBA")


SKIP_REASONS = Counter()
SWATCH_NAME = re.compile(r"^(l\d+|ai\d+)\b", re.I)   # finish/material swatch files like "L139.png", "AI01- Chrome.webp"


def build_cutout(path: str, is_art: bool):
    # Returns the cached cutout path, or None (and counts why) if the image is not a usable product photo.
    p = Path(path)
    if SWATCH_NAME.match(p.stem):
        SKIP_REASONS["finish swatch (by file name)"] += 1
        return None
    out = CUTOUT_DIR / (re.sub(r"[^A-Za-z0-9]+", "_", str(p.relative_to(IMAGES_DIR))) + ".png")
    if out.exists():
        return str(out)
    try:
        with Image.open(p) as im:
            cut = matte_white_background(im)
        alpha = np.array(cut)[..., 3]
        visible = (alpha > 127).mean()
        if visible < 0.02:
            SKIP_REASONS["almost all background"] += 1
            return None
        if not is_art and visible > 0.93:
            SKIP_REASONS["no studio background (swatch / close-up)"] += 1
            return None
        rgb = np.array(cut.convert("RGB")).astype(float)
        if not is_art and rgb[alpha > 127].mean() > 240:      # mostly white pixels: a line drawing, not a photo
            SKIP_REASONS["line drawing (white on white)"] += 1
            return None
        cut.crop(cut.getchannel("A").getbbox()).save(out)
        return str(out)
    except Exception as e:                        # never let one bad file stop the run
        SKIP_REASONS[f"error: {type(e).__name__}"] += 1
        return None


t0 = time.time()
df["cutout"] = [build_cutout(p, a) for p, a in zip(df["path"], df["is_art"])]
before = len(df)
df = df[df["cutout"].notna()].reset_index(drop=True)      # only clean product cutouts are trained on
print(f"{len(df)}/{before} images kept after cutout ({time.time() - t0:.0f}s). Dropped: {dict(SKIP_REASONS) or 'none'}")

WALLS = [(236, 231, 222), (222, 214, 200), (205, 210, 205), (196, 187, 175), (233, 224, 214),
         (94, 98, 96), (58, 56, 60), (170, 150, 128), (215, 220, 224)]
FLOORS = [(176, 140, 100), (140, 105, 75), (205, 190, 165), (95, 80, 70), (160, 150, 140), (60, 55, 52)]


def make_room_background(rng: random.Random, size: int = 512):
    wall, floor = rng.choice(WALLS), rng.choice(FLOORS)
    horizon = int(size * rng.uniform(0.58, 0.68))
    img = Image.new("RGB", (size, size), wall)
    d = ImageDraw.Draw(img)
    d.rectangle([0, horizon, size, size], fill=floor)
    d.rectangle([0, horizon - 10, size, horizon], fill=tuple(min(255, c + 25) for c in wall))   # baseboard
    if rng.random() < 0.5:                                                                      # window
        x0 = rng.randint(int(size * 0.08), int(size * 0.55)); w = rng.randint(90, 150)
        d.rectangle([x0, int(size * 0.12), x0 + w, horizon - 60], fill=(250, 250, 244), outline=(120, 120, 110), width=4)
    grad = np.linspace(1.06, 0.92, size)[:, None, None]
    return Image.fromarray(np.clip(np.array(img) * grad, 0, 255).astype("uint8")), horizon


def compose_room_sample(cut: Image.Image, is_art: bool, rng: random.Random, size: int = 512):
    bg, horizon = make_room_background(rng, size)
    cw, ch = cut.size
    if is_art:
        scale = rng.uniform(0.28, 0.5) * size / max(cw, ch)
    else:
        scale = rng.uniform(0.45, 0.85) * size / max(cw, ch)
    cut = cut.resize((max(1, int(cw * scale)), max(1, int(ch * scale))), Image.Resampling.LANCZOS)
    cw, ch = cut.size
    x = int(size / 2 - cw / 2 + rng.uniform(-0.12, 0.12) * size)
    if is_art:
        y = int(horizon * rng.uniform(0.12, 0.35))
    else:
        y = min(size - ch, int(horizon + (size - horizon) * rng.uniform(0.15, 0.55)) - ch)
        shadow = Image.new("RGBA", bg.size, (0, 0, 0, 0))
        ImageDraw.Draw(shadow).ellipse([x, y + ch - int(ch * 0.08), x + cw, y + ch + int(ch * 0.1)], fill=(0, 0, 0, 90))
        bg = Image.alpha_composite(bg.convert("RGBA"), shadow.filter(ImageFilter.GaussianBlur(9))).convert("RGB")
    bg.paste(cut, (x, max(0, y)), cut)
    return bg


def plain_product_sample(cutout_path: str, size: int = 512):
    with Image.open(cutout_path) as im:
        im = im.convert("RGBA")
        flat = Image.alpha_composite(Image.new("RGBA", im.size, (255, 255, 255, 255)), im).convert("RGB")
    flat.thumbnail((size, size), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (size, size), (255, 255, 255))
    canvas.paste(flat, ((size - flat.width) // 2, (size - flat.height) // 2))
    return canvas


def make_sample(row, rng: random.Random, size: int = 512):
    """Returns (PIL image, caption): a product pasted into a synthetic room, or a plain white-background product photo."""
    label = f"{row['name']}" if row["category"] in row["name"].lower() else f"{row['name']} {row['category']}"
    if rng.random() < P_ROOM_BACKGROUND:
        room = rng.choice(row["rooms"])
        with Image.open(row["cutout"]) as c:
            image = compose_room_sample(c.convert("RGBA"), row["is_art"], rng, size)
        caption = f"{TRIGGER}, a {label} in a {room}, interior design photo"
    else:
        image = plain_product_sample(row["cutout"], size)
        caption = f"{TRIGGER}, product photo of a {label}, white background"
    return image, caption


In [ ]:
import matplotlib.pyplot as plt

rng_preview = random.Random(SEED)
sample_rows = df.sample(min(8, len(df)), random_state=SEED)
fig, axes = plt.subplots(2, 4, figsize=(16, 9))
for ax, (_, row) in zip(axes.ravel(), sample_rows.iterrows()):
    img, cap = make_sample(row, rng_preview)
    ax.imshow(img); ax.axis("off"); ax.set_title(cap[:48] + "...", fontsize=8)
fig.suptitle("Training samples (background-removed products pasted into synthetic rooms) - same data as the SD1.5 run")
fig.tight_layout(); plt.show()


## 5. Write the dataset the way the training script expects it

The SD1.5 notebook generated training pairs on the fly, inside its own training loop (a fresh random room every
time an image was seen). Hugging Face's Qwen-Image LoRA script expects a plain folder of images plus a
`metadata.jsonl` mapping each file to its caption, so this cell materializes `TRAIN_STEPS`-worth of samples to disk
once, up front, instead. This also means each training "epoch" in the script's own terms sees a fixed set of
room compositions rather than infinite fresh ones — a reasonable trade for being able to use a tested, maintained
training script instead of a hand-rolled loop.


In [ ]:
import random as _random

DATASET_DIR = WORK_DIR / "qwen_dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Roughly enough unique samples to cover MAX_TRAIN_STEPS * BATCH_SIZE without too much exact repetition -
# the room/scale/position randomization in make_sample() still gives each repeat visible variety.
N_SAMPLES = max(len(df), min(4000, MAX_TRAIN_STEPS * BATCH_SIZE * 2))
rng_ds = random.Random(SEED)

metadata_path = DATASET_DIR / "metadata.jsonl"
written = 0
with open(metadata_path, "w") as f:
    for i in range(N_SAMPLES):
        row = df.iloc[i % len(df)]
        try:
            image, caption = make_sample(row, rng_ds, RESOLUTION)
        except Exception as e:
            print(f"skipping sample {i} ({row['name']}): {type(e).__name__}: {e}")
            continue
        fname = f"sample_{i:05d}.jpg"
        image.convert("RGB").save(DATASET_DIR / fname, quality=92)
        f.write(json.dumps({"file_name": fname, "text": caption}) + "\n")
        written += 1

print(f"wrote {written} training images + captions to {DATASET_DIR}")


## 6. Is this actually feasible on your Colab plan? Read this before Section 8 runs.

Qwen-Image's transformer is roughly 20-25x larger than SD1.5's UNet. LoRA only *trains* a small number of adapter
weights, but every step still runs a full forward and backward pass through the entire frozen transformer to get
there — so the per-step cost scales with the full 20B, not with the (tiny) number of trainable parameters.

**Honest estimate, not measured in this environment:** on a single A100 (40GB), 4-bit-quantized, batch size 1,
512px, community-reported Qwen-Image LoRA fine-tunes run roughly **8-20 seconds per optimizer step**. At that rate:

| Steps | ~Time (8s/step) | ~Time (20s/step) |
|---|---|---|
| 300 | 40 min | 100 min |
| 600 (this notebook's default) | 80 min | 200 min (3h20m) |
| 1200 (SD1.5's default) | 160 min (2h40m) | 400 min (6h40m) |

This is wide because it genuinely depends on your assigned GPU, Colab's current load, and image resolution. The
next cell measures your *actual* speed from a short warmup before the real run starts, and prints an updated
estimate for your configured `MAX_TRAIN_STEPS` — read that number before you walk away from this notebook for a
few hours. Colab sessions (even Pro+) can disconnect; `SAVE_EVERY` is set low specifically so a disconnect costs
you less than an hour of progress, and you can resume from the last checkpoint rather than starting over.

**If you're on anything less than an A100 (a T4, L4, or free tier), stop here.** The next cell will tell you this
directly and raise rather than let you discover it 20 minutes into a run via an out-of-memory crash.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit("No GPU found. In Colab: Runtime > Change runtime type > A100 GPU, then run all again.")

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU: {gpu_name}  ({total_vram_gb:.0f} GB)")

MIN_VRAM_GB = 38  # a 40GB A100 reports slightly under 40 once you account for reserved memory
if total_vram_gb < MIN_VRAM_GB:
    raise SystemExit(
        f"This GPU reports {total_vram_gb:.0f} GB. Qwen-Image's 20B transformer, even 4-bit quantized, needs "
        f"an A100 (40GB) or better — a T4/L4 (15-24GB) will out-of-memory partway through training, after already "
        f"spending time downloading the ~40GB of model weights. In Colab: Runtime > Change runtime type > A100 GPU "
        f"(requires Colab Pro or Pro+). This check exists so you find out now, not after a long download and a "
        f"partial run."
    )
print("GPU check passed - continuing.")


## 7. Install `diffusers` from source and locate the Qwen-Image LoRA training script

Qwen-Image support is recent enough that it may not be in the `diffusers` version Colab ships by default, so this
installs from source and clones the repository to get its `examples/` scripts. The exact script name has moved
around across `diffusers` releases as Qwen-Image support matured, so this searches for it rather than assuming one
exact path, and fails with a clear message (not a wrong guess) if none of the expected names exist in the version
you end up with.


In [ ]:
import subprocess, glob, sys as _sys

!pip install -q -U git+https://github.com/huggingface/diffusers
!pip install -q -U transformers accelerate peft bitsandbytes sentencepiece

if not Path("diffusers").is_dir():
    !git clone -q --depth 1 https://github.com/huggingface/diffusers.git

candidates = sorted(set(
    glob.glob("diffusers/examples/dreambooth/*qwen*image*.py")
    + glob.glob("diffusers/examples/dreambooth/*qwenimage*.py")
    + glob.glob("diffusers/examples/text_to_image/*qwen*image*.py")
    + glob.glob("diffusers/examples/*/*qwen*image*.py")
))
print("candidate training scripts found:", candidates or "(none)")

if not candidates:
    available = sorted(glob.glob("diffusers/examples/*/train_*lora*.py"))
    raise SystemExit(
        "Could not find a Qwen-Image LoRA training script in this diffusers checkout under examples/. "
        "diffusers' example scripts for newer models move between release cycles, so this is a real 'needs_input', "
        "not something to guess past.\n\n"
        "Other LoRA training scripts that DO exist in this checkout (for reference, to see the current naming "
        f"pattern):\n  " + "\n  ".join(available[:20]) + "\n\n"
        "Check https://github.com/huggingface/diffusers/tree/main/examples for the current Qwen-Image script name "
        "and update TRAIN_SCRIPT below by hand."
    )

TRAIN_SCRIPT = candidates[0]
print("using:", TRAIN_SCRIPT)
!python {TRAIN_SCRIPT} --help 2>&1 | head -5


## 8. Train

Runs Hugging Face's own maintained script via `accelerate launch`, with the settings from the CONFIG cell. This
streams the script's real output live — including whatever loss/step logging it prints natively — rather than this
notebook re-parsing or re-plotting it, since that native output is the ground truth for what's actually happening.

The first ~10 steps are the slowest (CUDA kernel compilation, cache warmup); the printed "steps/sec" a few dozen
steps in is a more honest read of the real rate than the first few steps alone.


In [ ]:
import time

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run_output_dir = OUTPUT_DIR / "final"

cmd = [
    "accelerate", "launch", TRAIN_SCRIPT,
    f"--pretrained_model_name_or_path={BASE_MODEL_ID}",
    f"--instance_data_dir={DATASET_DIR}",
    f"--output_dir={run_output_dir}",
    f"--instance_prompt={TRIGGER}",
    f"--resolution={RESOLUTION}",
    f"--train_batch_size={BATCH_SIZE}",
    f"--max_train_steps={MAX_TRAIN_STEPS}",
    f"--learning_rate={LEARNING_RATE}",
    f"--rank={LORA_RANK}",
    f"--checkpointing_steps={SAVE_EVERY}",
    f"--seed={SEED}",
    "--mixed_precision=bf16",
    "--use_8bit_adam",
]
if GRADIENT_CHECKPOINTING:
    cmd.append("--gradient_checkpointing")
if QUANTIZE_4BIT:
    # Flag name varies by diffusers version for this script; if this specific flag errors, check
    # `python {TRAIN_SCRIPT} --help` output above for the current quantization argument and edit this list.
    cmd.append("--quantize=nf4")

print("running:", " ".join(cmd))
t0 = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
elapsed_min = (time.time() - t0) / 60
print(f"\n--- training process exited with code {proc.returncode} after {elapsed_min:.1f} min ---")
if proc.returncode != 0:
    raise SystemExit(
        f"Training script exited with a non-zero code ({proc.returncode}). Scroll up to the actual error - a "
        f"common first failure on an unfamiliar script is a renamed/removed CLI flag; re-run the --help cell "
        f"above and compare against the `cmd` list in this cell."
    )

with open(OUTPUT_DIR / "training_config.json", "w") as f:
    json.dump(dict(base_model=BASE_MODEL_ID, trigger=TRIGGER, steps=MAX_TRAIN_STEPS, rank=LORA_RANK,
                   resolution=RESOLUTION, batch_size=BATCH_SIZE, lr=LEARNING_RATE, images=int(len(df)),
                   suppliers=sorted(df["supplier"].unique().tolist()), elapsed_minutes=round(elapsed_min, 1),
                   train_script=TRAIN_SCRIPT), f, indent=2)
print("done - LoRA weights are in", run_output_dir)


## 9. Save for download

Same as the SD1.5 notebook: zip the final LoRA for a one-click download.

In [ ]:
zip_path = shutil.make_archive(str(OUTPUT_DIR / "curalina_qwen_lora"), "zip", str(run_output_dir))
print("zipped to", zip_path)
if IN_COLAB:
    from google.colab import files
    print("Find it in Drive at:", zip_path, "\n(or uncomment the next line to trigger a direct browser download)")
    # files.download(zip_path)


## 10. Use it: plain model vs. model + your LoRA

Loads the base pipeline once, generates a room with the plain model, then loads your LoRA weights with
`pipe.load_lora_weights(...)` (a stable, standard `diffusers` API — the same call the SD1.5 notebook uses) and
generates the same prompt again for comparison. This part of the notebook is on solid, well-documented ground
regardless of how training itself went, since loading a trained LoRA for inference is the same mechanism across
every diffusers pipeline.


In [ ]:
from diffusers import QwenImagePipeline
import torch as _torch

compare_prompt = f"{TRIGGER}, a sofa and a coffee table in a bright living room, interior design photo"

pipe_kwargs = dict(torch_dtype=_torch.bfloat16)
if QUANTIZE_4BIT:
    from transformers import BitsAndBytesConfig
    pipe_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4")

pipe = QwenImagePipeline.from_pretrained(BASE_MODEL_ID, **pipe_kwargs).to(device)

gen = _torch.Generator(device=device).manual_seed(SEED)
base_image = pipe(compare_prompt, num_inference_steps=28, generator=gen).images[0]

pipe.load_lora_weights(str(run_output_dir))
gen = _torch.Generator(device=device).manual_seed(SEED)
lora_image = pipe(compare_prompt, num_inference_steps=28, generator=gen).images[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(base_image); axes[0].axis("off"); axes[0].set_title("Base Qwen-Image")
axes[1].imshow(lora_image); axes[1].axis("off"); axes[1].set_title("Base + your LoRA")
fig.suptitle(compare_prompt)
fig.tight_layout(); plt.show()


## Using the result later

Same deal as the SD1.5 LoRA: point the room-rendering service at the downloaded weights (an equivalent
`SD15_LORA_PATH`-style env var for whichever renderer ends up serving Qwen-Image, once one exists — there is no
`qwen_renderer` service in this repo yet; see `ai_services/renderers/sd15/` for the pattern a new one would follow)
and set the trigger word (`crlnstyle`) in your prompts, the same way `SD15_LORA_TRIGGER` works today.
